# Advanced Machine Learning 2026/2027
## Project: Modeling Recovery after Lower-Limb Musculoskeletal Injury
### M2 — Bayesian Networks

**Group:** 14

| Number | Name |
|---|---|
| fc68043 | David Belykh |
| fc68080 | Matilde Quitério |
| fc68091 | Ramyad Raadi |

### Statement on the use of generative AI
The generative AI tool **ChatGPT (OpenAI)** was used to support notebook structuring, code generation/debugging, clarification of Bayesian Network concepts, and improvement of written explanations. All submitted content, results, methodological choices, and interpretations were reviewed and validated by the group members before submission.

## 0. Setup and data loading

In this milestone, each row of `rehabilitation.csv` is treated as an independent **patient-week** observation, as required by the project statement.

We merge the weekly rehabilitation data with baseline patient information using `patient_id`, because the Bayesian Network includes the baseline variable `risk_score`.

> If `pgmpy` is not installed, run `%pip install pgmpy networkx` once and restart the kernel.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

STATE_ORDER = ["Limited", "Recovering", "Functional"]
INTERVENTION_ORDER = [
    "Rest/Recovery",
    "Light Exercise",
    "Standard Therapy",
    "Intensive Therapy",
]

def find_file(filename, search_paths=(".", "..", "/mnt/data")):
    for base in search_paths:
        if not os.path.exists(base):
            continue
        for root, _, files in os.walk(base):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(
        f"Could not find {filename}. Place this notebook in the project folder "
        "or make sure the dataset directory is accessible."
    )

patients_path = find_file("patients.csv")
rehab_path = find_file("rehabilitation.csv")

patients = pd.read_csv(patients_path)
rehab = pd.read_csv(rehab_path)

data = rehab.merge(
    patients[["patient_id", "risk_score"]],
    on="patient_id",
    how="left",
    validate="many_to_one"
)

print("patients.csv:", patients_path)
print("rehabilitation.csv:", rehab_path)
print("Merged shape:", data.shape)
display(data.head())

In [ ]:
required_columns = [
    "patient_id",
    "risk_score",
    "state",
    "mobility",
    "pain",
    "fatigue",
    "functional_confidence",
    "activity",
    "adherence",
    "intervention",
    "utility",
]

missing_columns = [c for c in required_columns if c not in data.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Missing values in selected variables:")
display(data[required_columns].isna().sum().to_frame("n_missing"))

# Task 1 — Structure design

The goal is to build a Bayesian Network that represents plausible probabilistic relationships between baseline risk, recovery state, observable rehabilitation measurements, treatment decisions, and outcome.

We use:

- `risk_score`
- `state`
- `mobility`
- `pain`
- `fatigue`
- `functional_confidence`
- `activity`
- `adherence`
- `intervention`
- `utility`

### Proposed structure

- `risk_score → state`
- `state → mobility`
- `state → pain`
- `state → fatigue`
- `state → functional_confidence`
- `state → activity`
- `state → adherence`
- `state → intervention`
- `state → utility`
- `intervention → utility`
- `adherence → utility`

### Rationale

`risk_score` summarizes baseline factors associated with a less favorable recovery trajectory, so it is modeled as influencing the recovery `state`.

The recovery `state` is modeled as a common cause of the observed rehabilitation measurements. A patient who is Limited, Recovering, or Functional is expected to show different mobility, pain, fatigue, confidence, activity, and adherence profiles.

`intervention` is modeled as depending on the current recovery state because rehabilitation decisions may differ according to patient condition.

Finally, `utility` depends on `state`, `intervention`, and `adherence`, reflecting recovery status, treatment burden, and adherence.

These arrows are **modeling assumptions** and should not be interpreted as proven causal relations.

In [ ]:
edges = [
    ("risk_score", "state"),
    ("state", "mobility"),
    ("state", "pain"),
    ("state", "fatigue"),
    ("state", "functional_confidence"),
    ("state", "activity"),
    ("state", "adherence"),
    ("state", "intervention"),
    ("state", "utility"),
    ("intervention", "utility"),
    ("adherence", "utility"),
]

bn_structure = DiscreteBayesianNetwork(edges)

print("Nodes:")
print(list(bn_structure.nodes()))

print("\nEdges:")
for edge in bn_structure.edges():
    print(edge)

In [ ]:
plt.figure(figsize=(13, 8))

G = nx.DiGraph(edges)

pos = {
    "risk_score": (0, 2.5),
    "state": (2, 2.5),
    "mobility": (4, 4.5),
    "pain": (4, 3.6),
    "fatigue": (4, 2.7),
    "functional_confidence": (4, 1.8),
    "activity": (4, 0.9),
    "adherence": (4, 0.0),
    "intervention": (4, -1.0),
    "utility": (7, 0.5),
}

nx.draw(
    G,
    pos=pos,
    with_labels=True,
    node_size=3500,
    font_size=9,
    arrowsize=20,
    edgecolors="black",
)

plt.title("Proposed Bayesian Network")
plt.axis("off")
plt.show()

# Task 2 — Discretization

The Bayesian Networks considered in the project are discrete, so the continuous variables must be transformed into a small number of categories.

We use three categories per continuous variable to keep the model interpretable and the CPTs manageable.

| Variable | Categories |
|---|---|
| `risk_score` | Low: < 33.3, Medium: 33.3–66.7, High: > 66.7 |
| `mobility` | Low: < 40, Medium: 40–70, High: > 70 |
| `pain` | Low: 0–3, Medium: >3–6, High: >6 |
| `fatigue` | Low: 0–3, Medium: >3–6, High: >6 |
| `functional_confidence` | Low: < 40, Medium: 40–70, High: > 70 |
| `activity` | Low: < 35, Medium: 35–70, High: > 70 |
| `adherence` | Low: < 60, Medium: 60–80, High: > 80 |

For `utility`, the statement does not provide fixed meaningful cutoffs, so we use **tertiles of the observed data** to obtain Low, Medium, and High outcome categories.

The exact utility thresholds are printed below.

In [ ]:
disc = data.copy()

disc["risk_score"] = pd.cut(
    disc["risk_score"],
    bins=[-np.inf, 33.3, 66.7, np.inf],
    labels=["Low", "Medium", "High"]
)

disc["mobility"] = pd.cut(
    disc["mobility"],
    bins=[-np.inf, 40, 70, np.inf],
    labels=["Low", "Medium", "High"]
)

disc["pain"] = pd.cut(
    disc["pain"],
    bins=[-np.inf, 3, 6, np.inf],
    labels=["Low", "Medium", "High"]
)

disc["fatigue"] = pd.cut(
    disc["fatigue"],
    bins=[-np.inf, 3, 6, np.inf],
    labels=["Low", "Medium", "High"]
)

disc["functional_confidence"] = pd.cut(
    disc["functional_confidence"],
    bins=[-np.inf, 40, 70, np.inf],
    labels=["Low", "Medium", "High"]
)

disc["activity"] = pd.cut(
    disc["activity"],
    bins=[-np.inf, 35, 70, np.inf],
    labels=["Low", "Medium", "High"]
)

disc["adherence"] = pd.cut(
    disc["adherence"],
    bins=[-np.inf, 60, 80, np.inf],
    labels=["Low", "Medium", "High"]
)

utility_q1, utility_q2 = disc["utility"].quantile([1/3, 2/3]).values

disc["utility"] = pd.cut(
    disc["utility"],
    bins=[-np.inf, utility_q1, utility_q2, np.inf],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

print("Utility cutoffs:")
print(f"Low    : utility <= {utility_q1:.3f}")
print(f"Medium : {utility_q1:.3f} < utility <= {utility_q2:.3f}")
print(f"High   : utility > {utility_q2:.3f}")

display(
    disc[
        [
            "risk_score", "state", "mobility", "pain", "fatigue",
            "functional_confidence", "activity", "adherence",
            "intervention", "utility"
        ]
    ].head(10)
)

In [ ]:
discrete_variables = [
    "risk_score",
    "state",
    "mobility",
    "pain",
    "fatigue",
    "functional_confidence",
    "activity",
    "adherence",
    "intervention",
    "utility",
]

for col in discrete_variables:
    print(f"\n--- {col} ---")
    counts = disc[col].value_counts(dropna=False)
    percentages = disc[col].value_counts(normalize=True, dropna=False) * 100
    display(pd.DataFrame({
        "count": counts,
        "percentage": percentages.round(2)
    }))

In [ ]:
print("Missing values after discretization:")
display(disc[discrete_variables].isna().sum().to_frame("n_missing"))

if disc[discrete_variables].isna().any().any():
    raise ValueError(
        "Discretization produced missing values. Check the variable ranges and bin limits."
    )

# Task 3 — Estimating CPTs by counting

The conditional probability tables are estimated using **relative frequencies**.

For a variable \(X\) with parents \(Pa(X)\):

\[
P(X=x \mid Pa(X)=p)
=
\frac{count(X=x, Pa(X)=p)}
{count(Pa(X)=p)}
\]

Each patient-week is treated as an independent observation.

In [ ]:
def conditional_probability_table(df, child, parents=None):
    # Compute counts and relative frequencies for a node and its parent configuration.
    parents = parents or []

    if len(parents) == 0:
        counts = (
            df.groupby(child, observed=False)
              .size()
              .rename("count")
              .reset_index()
        )
        total = counts["count"].sum()
        counts["parent_total"] = total
        counts["probability"] = counts["count"] / total
        return counts

    group_cols = parents + [child]

    counts = (
        df.groupby(group_cols, observed=False)
          .size()
          .rename("count")
          .reset_index()
    )

    counts["parent_total"] = counts.groupby(
        parents, observed=False
    )["count"].transform("sum")

    counts["probability"] = np.where(
        counts["parent_total"] > 0,
        counts["count"] / counts["parent_total"],
        np.nan
    )

    return counts

### 3.1 Prior distribution of `risk_score`

In [ ]:
cpt_risk = conditional_probability_table(disc, "risk_score")
display(cpt_risk)

### 3.2 Recovery state conditioned on risk

This estimates \(P(state \mid risk\_score)\).

We use the resulting probabilities to inspect whether high baseline risk is associated with a greater probability of `Limited`, and whether lower-risk observations show more favorable recovery states.

In [ ]:
cpt_state = conditional_probability_table(
    disc,
    child="state",
    parents=["risk_score"]
)

display(cpt_state)

state_given_risk = pd.crosstab(
    disc["risk_score"],
    disc["state"],
    normalize="index"
).reindex(columns=STATE_ORDER)

display(state_given_risk.round(3))

In [ ]:
state_given_risk.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5)
)

plt.ylabel("Probability")
plt.title("P(state | risk_score)")
plt.legend(title="state")
plt.tight_layout()
plt.show()

### 3.3 Observable measurements conditioned on recovery state

We inspect the distributions of the six observable rehabilitation measurements given `state`.

In [ ]:
for variable in [
    "mobility",
    "pain",
    "fatigue",
    "functional_confidence",
    "activity",
    "adherence",
]:
    print(f"\n===== P({variable} | state) =====")
    table = pd.crosstab(
        disc["state"],
        disc[variable],
        normalize="index"
    ).reindex(index=STATE_ORDER)
    display(table.round(3))

The expected qualitative pattern is that more favorable recovery states should generally be associated with higher mobility, confidence, and activity, and with lower pain and fatigue.

The final interpretation must be based on the actual probabilities printed above.

### 3.4 Intervention conditioned on recovery state

This estimates \(P(intervention \mid state)\).

In [ ]:
cpt_intervention = conditional_probability_table(
    disc,
    child="intervention",
    parents=["state"]
)

display(cpt_intervention)

intervention_given_state = pd.crosstab(
    disc["state"],
    disc["intervention"],
    normalize="index"
).reindex(index=STATE_ORDER, columns=INTERVENTION_ORDER)

display(intervention_given_state.round(3))

In [ ]:
intervention_given_state.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 5)
)

plt.ylabel("Probability")
plt.title("P(intervention | state)")
plt.legend(title="intervention", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 3.5 Utility with multiple parents

In the proposed network, `utility` has three parents:

- `state`
- `intervention`
- `adherence`

Therefore we estimate:

\[
P(utility \mid state, intervention, adherence)
\]

This explicitly computes frequencies for all combinations of the parent values.

In [ ]:
cpt_utility = conditional_probability_table(
    disc,
    child="utility",
    parents=["state", "intervention", "adherence"]
)

display(cpt_utility.head(60))

print("Number of rows in the full utility CPT:", len(cpt_utility))

In [ ]:
utility_parent_counts = (
    disc.groupby(
        ["state", "intervention", "adherence"],
        observed=False
    )
    .size()
    .rename("n")
    .reset_index()
    .sort_values("n")
)

display(utility_parent_counts.head(15))

print(
    "Smallest number of observations in a "
    "state/intervention/adherence configuration:",
    utility_parent_counts["n"].min()
)

### Fit complete CPDs with pgmpy

`MaximumLikelihoodEstimator` estimates the CPDs from observed relative frequencies. We use it after explicitly inspecting the counts above so the network has a complete set of probability tables.

In [ ]:
model_variables = [
    "risk_score",
    "state",
    "mobility",
    "pain",
    "fatigue",
    "functional_confidence",
    "activity",
    "adherence",
    "intervention",
    "utility",
]

bn_data = disc[model_variables].copy()

for col in bn_data.columns:
    bn_data[col] = bn_data[col].astype(str)

bn = DiscreteBayesianNetwork(edges)

bn.fit(
    bn_data,
    estimator=MaximumLikelihoodEstimator
)

print("Model valid:", bn.check_model())
print("Number of fitted CPDs:", len(bn.get_cpds()))

In [ ]:
for variable in [
    "state",
    "mobility",
    "pain",
    "intervention",
    "utility",
]:
    print(f"\n===== CPD: {variable} =====")
    print(bn.get_cpds(variable))

### CPT discussion

After execution, check directly from the output:

1. Does a high `risk_score` increase the probability of being `Limited`?
2. Are `Functional` patient-weeks more likely to have high `mobility` and low `pain`?
3. Are different interventions more common in different recovery states?
4. How does `utility` vary across `state`, `intervention`, and `adherence`?
5. Are some CPT estimates based on very small counts?

The final report should refer to the actual probabilities obtained.

# Task 4 — Conditional independence analysis

The proposed graph implies several conditional independencies.

### Mobility and risk score

Path:

`risk_score → state → mobility`

Once `state` is known, the path is blocked:

\[
mobility \perp risk\_score \mid state
\]

### Mobility and pain

Path:

`mobility ← state → pain`

Conditioning on `state` blocks their common-cause path:

\[
mobility \perp pain \mid state
\]

### Mobility and intervention

Path:

`mobility ← state → intervention`

Therefore:

\[
mobility \perp intervention \mid state
\]

These are implications of our chosen network structure. They are not automatically proven by the dataset.

In [ ]:
independencies = bn_structure.get_independencies()

print("Sample of independencies implied by the network:\n")
indep_text = str(independencies)
print(indep_text[:8000])

In [ ]:
def show_dconnection(model, x, y, observed=None):
    observed = observed or []
    connected = model.is_dconnected(x, y, observed=observed)
    evidence_text = ", ".join(observed) if observed else "no evidence"

    print(
        f"{x} and {y} | {evidence_text}: "
        f"{'d-connected' if connected else 'd-separated'}"
    )

show_dconnection(bn_structure, "mobility", "risk_score")
show_dconnection(bn_structure, "mobility", "risk_score", observed=["state"])

show_dconnection(bn_structure, "mobility", "pain")
show_dconnection(bn_structure, "mobility", "pain", observed=["state"])

show_dconnection(bn_structure, "mobility", "intervention")
show_dconnection(bn_structure, "mobility", "intervention", observed=["state"])

show_dconnection(bn_structure, "risk_score", "utility")
show_dconnection(bn_structure, "risk_score", "utility", observed=["state"])

### Interpretation of the independence analysis

Conditional independence is determined by the network structure we designed.

For instance, by making `state` the direct parent of both `mobility` and `pain`, we assume that once recovery state is known, there is no additional direct probabilistic dependency between mobility and pain in this model.

This is a simplification. In a real rehabilitation process, pain and mobility could remain related even after accounting for an overall recovery state.

# Task 5 — Interpretation

## Which measurements are most informative about recovery state?

Compare the conditional distributions \(P(measurement \mid state)\).

A variable is potentially more informative when its category probabilities differ strongly between `Limited`, `Recovering`, and `Functional`.

For example, if `mobility=High` is common in `Functional` observations and rare in `Limited` observations, mobility carries useful information about recovery state.

The final answer should cite the actual probabilities produced by this notebook.

## Agreement with M0

M0 explored how mobility, pain, fatigue, confidence, activity, adherence, interventions, and utility vary across recovery states.

M2 represents these relationships more formally using conditional probabilities.

## Main assumptions

The proposed network assumes that:

- baseline `risk_score` influences the weekly recovery `state`;
- weekly measurements depend directly on `state`;
- the measurements are conditionally independent of each other once `state` is known;
- intervention selection depends directly on `state`;
- `utility` depends directly on `state`, `intervention`, and `adherence`;
- patient-weeks are independent observations;
- discretization is an approximation of the original continuous measurements.

## Assumptions versus data-supported quantities

The **network structure and arrow directions** are modeling choices.

The **CPT values** are estimated from the data and can be inspected to determine whether the numerical patterns agree with our expectations.

For example, `risk_score → state` is an assumption of the graph, while the estimated values of \(P(state \mid risk\_score)\) show how strongly the two variables are associated in the dataset.

## Not a causal model

The network describes probabilistic dependencies and conditional independencies.

An arrow such as `state → intervention` is not proof that recovery state causally determines treatment.

Similarly, differences in \(P(utility \mid state, intervention, adherence)\) do not prove that one intervention causes a better rehabilitation outcome.

The dataset is synthetic and the structure was designed rather than causally identified.

# Final reflection

This Bayesian Network provides a compact probabilistic representation of the rehabilitation problem:

- baseline risk is related to recovery state;
- recovery state is reflected in observable measurements;
- intervention selection varies with recovery state;
- utility is related to recovery state, intervention, and adherence.

Compared with M1, the goal is no longer to discover unlabeled clusters. In M2, we explicitly specify relationships among known variables and estimate their conditional probability tables.

A major limitation is that each patient-week is still treated independently. The model does not represent the temporal ordering of consecutive weeks from the same patient. Temporal dependencies will be addressed later with Hidden Markov Models.

## Submission checklist

- [ ] Run all cells from top to bottom.
- [ ] Confirm the BN graph is displayed correctly.
- [ ] Inspect the discretized category frequencies.
- [ ] Record the exact utility thresholds.
- [ ] Replace generic comments with specific probabilities from the output where appropriate.
- [ ] Check sparse CPT configurations.
- [ ] Confirm the d-separation cells run with your installed `pgmpy` version.
- [ ] Keep all outputs visible.
- [ ] Review the AI-use statement.
- [ ] Save as **`M2_G14.ipynb`**.